# OpenPack Temporal VLM Fine-Tuning
**Live Notebook Link:** [INSERT YOUR PUBLIC KAGGLE LINK HERE BEFORE SUBMISSION]
**Environment:** Kaggle 2xT4 GPUs (32GB Total)


In [ ]:
# VRAM Budget Calculation (Required Format)
model_base_4bit    = 2.0   # GB — Your model at 4-bit
lora_adapters      = 0.3   # GB — LoRA rank + alpha
frames_per_clip    = 8     # Sampled frames per 5-sec clip
frame_tokens       = 256   # Visual tokens per frame (after vision encoder)
batch_size         = 2
token_hidden_dim   = 1536  # Your model's hidden size
activation_gb      = (frames_per_clip * frame_tokens * batch_size * token_hidden_dim * 2) / 1e9
# With gradient checkpointing: activation_gb * 0.4 (recomputed, not stored)
total_vram_gb      = model_base_4bit + lora_adapters + (activation_gb * 0.4)
print(f"Estimated VRAM: {total_vram_gb:.2f} GB")
# Must be < 16 GB for T4 or < 40 GB for A100


In [ ]:
!pip install -q -U transformers accelerate bitsandbytes peft trl webdataset qwen-vl-utils


In [ ]:
import os
import torch
import webdataset as wds
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

MODEL_ID = "Qwen/Qwen2.5-VL-2B-Instruct"

# 1. 4-bit Quantization Configuration (T4 Optimized)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16  # fp16 for T4 Turing architecture
)

# 2. Load Base Model & Processor
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    device_map="auto",
    quantization_config=bnb_config
)

# 3. CRITICAL: Gradient Flow Patch for Video/Image inputs
model.enable_input_require_grads()
model.gradient_checkpointing_enable()

# 4. LoRA Configuration
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# 5. WebDataset Loader (Streaming to prevent RAM spikes)
def process_wds_sample(sample):
    # In full training, parse sample['json'] + 8 JPEG frames and convert to model inputs.
    return sample

# Dummy dataset for pipeline validation before OpenPack is fully downloaded
dummy_dataset = [
    {"input_ids": torch.randint(0, 1000, (128,)), "attention_mask": torch.ones(128)}
    for _ in range(100)
]

# Example real dataset hook (replace when OpenPack shards are mounted):
# train_dataset = wds.WebDataset('shards/openpack_train_0000.tar').map(process_wds_sample)

# 6. Aggressive Memory-Optimized Training Arguments
training_args = TrainingArguments(
    output_dir="./qwen-openpack-checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    learning_rate=2e-5,
    fp16=True,
    max_steps=200,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    remove_unused_columns=False,
    report_to="none"
)

# 7. SFT Trainer Initialization
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dummy_dataset,  # Replace with WebDataset stream
    peft_config=lora_config,
    tokenizer=processor.tokenizer,
)

print("Starting strict memory-constrained fine-tuning...")
# trainer.train(resume_from_checkpoint=False)  # Uncomment on Kaggle/GCP
